# Cbow Embeding Pta Manajemen Preprocesed

In [1]:
%%capture
!pip install plotly
!pip install --upgrade gensim

## Ringkasan Alur
- Import pustaka yang dibutuhkan
- Baca dataset `hasil_preprocessing_pta_manajemen.csv`
- Pembersihan teks (lowercase, hapus tanda baca/tag/digit)
- Tokenisasi dan pembuatan `corpus`
- Latih Word2Vec (CBOW) untuk embedding kata
- Hitung mean embedding per dokumen (rata-rata vektor kata)
- Bentuk DataFrame fitur `f1..f56` dan tambahkan label `spam`
- (Opsional) Simpan hasil ke CSV


In [2]:
from gensim.models import Word2Vec, FastText
import pandas as pd
import re

from sklearn.decomposition import PCA

from matplotlib import pyplot as plt
import plotly.graph_objects as go

import numpy as np

import warnings
warnings.filterwarnings('ignore')

# Load dataset TF-IDF berita dari file CSV
# Pastikan file 'hasil_tfidf_berita.csv' ada di direktori kerja notebook ini

df = pd.read_csv('hasil_preprocessing_pta_manajemen.csv')

In [3]:
from gensim.models import Word2Vec

In [4]:
import numpy as np

class MyTokenizer:
    def fit_transform(self, texts):
        # Tokenisasi sederhana: lowercase + split
        return [str(text).lower().split() for text in texts]

class MeanEmbeddingVectorizer:
    def __init__(self, word2vec_model):
        self.word2vec = word2vec_model
        # Perbaikan: gunakan vector_size (Gensim ≥ 4.0)
        self.dim = word2vec_model.wv.vector_size

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_tokenized = MyTokenizer().fit_transform(X)
        embeddings = []
        for words in X_tokenized:
            # Ambil vektor hanya untuk kata yang ada di vocab
            valid_vectors = [
                self.word2vec.wv[word] for word in words
                if word in self.word2vec.wv
            ]
            if valid_vectors:
                embeddings.append(np.mean(valid_vectors, axis=0))
            else:
                embeddings.append(np.zeros(self.dim))
        return np.array(embeddings)

    def fit_transform(self, X, y=None):
        return self.transform(X)

In [5]:
# Bangun korpus token dari teks hasil preprocessing: setiap baris -> list kata
# Gunakan kolom 'hasil_preprocessing' jika ada, jika tidak cari alternatif umum
import ast

text_col = 'hasil_preprocessing' if 'hasil_preprocessing' in df.columns else None
if text_col is None:
    possible_text_cols = ['clean', 'text', 'preprocessed', 'kalimat', 'sentence']
    text_col = next((c for c in possible_text_cols if c in df.columns), None)
if text_col is None:
    raise RuntimeError("Kolom teks tidak ditemukan. Pastikan 'hasil_preprocessing' atau kolom teks lain tersedia di CSV.")

# Parser aman: dukung format list-string seperti "['kata', 'kata2']" atau string biasa
def to_tokens(value):
    if isinstance(value, list):
        return [str(tok).lower() for tok in value if str(tok).strip()]
    if isinstance(value, str):
        s = value.strip()
        if s.startswith('[') and s.endswith(']'):
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, list):
                    return [str(tok).lower() for tok in parsed if str(tok).strip()]
            except Exception:
                pass
        return [tok for tok in s.lower().split() if tok]
    return []

texts = df[text_col].fillna("").tolist()
corpus = [tokens for tokens in (to_tokens(v) for v in texts) if tokens]

if len(corpus) == 0:
    raise RuntimeError("Corpus kosong: kolom teks ada tetapi tidak berisi token valid.")

# Contoh token dokumen pertama
corpus[0:1]

[['aplikasi',
  'nyata',
  'manfaat',
  'teknologi',
  'informasi',
  'komunikasi',
  'bidang',
  'layan',
  'administrasi',
  'akademik',
  'guru',
  'salah',
  'satu',
  'sistem',
  'portal',
  'akademik',
  'universitas',
  'trunojoyo',
  'madura',
  'implementasi',
  'proses',
  'selenggara',
  'temu',
  'kendala',
  'teknis',
  'non',
  'teknis',
  'teliti',
  'tuju',
  'puas',
  'langgan',
  'dasar',
  'analisis',
  'indeks',
  'puas',
  'langgan',
  'tinjau',
  'webqual',
  'fokus',
  'baik',
  'mutu',
  'layan',
  'website',
  'portal',
  'akademik',
  'universitas',
  'trunjoyo',
  'madura',
  'dasar',
  'importance',
  'performance',
  'analysis',
  'tinjau',
  'webqual',
  'teliti',
  'teliti',
  'kuantitaif',
  'deskriptif',
  'gambar',
  'deskripsi',
  'lukis',
  'sistematis',
  'faktual',
  'akurat',
  'kualitas',
  'layan',
  'jasa',
  'online',
  'mahasiswasampelpenelitianiniadalahmahasiswaangkatan',
  'teknik',
  'sampel',
  'teknik',
  'stratified',
  'random',
  'sam

## Tokenisasi dan Pelatihan Word2Vec (CBOW)
- Tokenisasi setiap baris `clean` menjadi list kata → `corpus`
- Latih model Word2Vec dengan default CBOW (`sg=0`) dan `vector_size=56`
- Hasil: embedding vektor untuk setiap kata di vocabulary


In [6]:
df.shape

(1026, 2)

In [7]:
# Latih model Word2Vec (CBOW by default: sg=0) dengan ukuran vektor 56
# Menggunakan korpus token yang dibangun dari fitur TF-IDF (nilai > 0)
model = Word2Vec(corpus, min_count=1, vector_size=56)

In [8]:
# (Opsional) contoh eksplorasi embeddings kata jika diperlukan
# model.wv.most_similar('eric')
# model.wv.most_similar_cosmul(positive=['phone', 'number'], negative=['call'])
# model.wv.doesnt_match("phone number prison cell".split())

# Simpan embeddings kata yang dilatih
filename = 'pta_embd.txt'
model.wv.save_word2vec_format(filename, binary=False)

In [9]:
# Mean embedding per dokumen: rata-rata vektor kata dari token TF-IDF
mean_embedding_vectorizer = MeanEmbeddingVectorizer(model)
# Gabungkan token menjadi string kalimat agar tokenizer bekerja sama seperti sebelumnya
joined_docs = [" ".join(tokens) for tokens in corpus]
mean_embedded = mean_embedding_vectorizer.fit_transform(joined_docs)

In [10]:
# Simpan vektor dokumen ke kolom 'array'
df['array']=list(mean_embedded)

## Rata-rata Embedding per Dokumen
- Gunakan `MeanEmbeddingVectorizer` untuk merata-ratakan vektor kata per dokumen
- Jika dokumen tidak punya kata di vocab, isi vektor nol berdimensi 56


In [11]:
df.head(5)

,abstrak_id,hasil_preprocessing,array
0,Aplikasi nyata pemanfaatan teknologi informasi...,"['aplikasi', 'nyata', 'manfaat', 'teknologi', ...","[-0.18787678, -0.12537281, -0.48232284, 0.5489..."
1,Tujuan penelitian ini adalah untuk mengetahui ...,"['tuju', 'teliti', 'persepsi', 'brand', 'assoc...","[-0.5575547, 0.15453357, -0.60565287, 0.569248..."
2,"ABSTRAK\n\nSatiyah, Pengaruh Faktor-faktor Pel...","['abstrak', 'satiyah', 'pengaruh', 'faktorfakt...","[-0.029168872, 0.08007342, -0.8495709, 0.74736..."
3,Abstrak\n\nPenelitian ini menggunakan metode k...,"['abstrak', 'teliti', 'metode', 'kuantitatif',...","[-0.03817744, 0.68616605, -0.8184872, 0.994374..."
4,Hasil dari penelitian ini dari perhitungan Cre...,"['hasil', 'teliti', 'hitung', 'credit', 'risk'...","[-0.58000076, 0.43820882, 0.13957301, 0.579606..."


In [12]:
df['embedding_length'] = df['array'].str.len()

In [13]:
print(df['embedding_length'])

0       56
1       56
2       56
3       56
4       56
        ..
1021    56
1022    56
1023    56
1024    56
1025    56
Name: embedding_length, Length: 1026, dtype: int64


## Bentuk DataFrame Fitur f1..f56 dan Tambah Label
- Ekstrak setiap dimensi embedding ke kolom `f1..f56`
- Tambahkan label `spam` dari dataset asli


In [14]:
df.shape

(1026, 4)

In [15]:
num_features = len(df['array'].iloc[0])  # asumsi semua list punya panjang sama
columns = [f'f{i+1}' for i in range(num_features)]

# Inisialisasi dictionary untuk menampung data per kolom
data_dict = {col: [] for col in columns}

# Looping setiap baris di kolom 'embedding'
for embedding_list in df['array']:
    for i, value in enumerate(embedding_list):
        data_dict[f'f{i+1}'].append(value)

# Buat DataFrame dari dictionary
embedding_df = pd.DataFrame(data_dict)

print(embedding_df)

            f1        f2        f3        f4        f5        f6        f7  \
0    -0.187877 -0.125373 -0.482323  0.548948 -0.051382 -0.167641  0.016427   
1    -0.557555  0.154534 -0.605653  0.569248 -0.223671  0.012987  0.120140   
2    -0.029169  0.080073 -0.849571  0.747365  0.095475 -0.209035 -0.108454   
3    -0.038177  0.686166 -0.818487  0.994374  0.099712 -0.285629  0.119406   
4    -0.580001  0.438209  0.139573  0.579606  0.347730 -0.519288  0.203503   
...        ...       ...       ...       ...       ...       ...       ...   
1021  0.075879  0.546321 -0.568065  1.203466  0.393896 -0.622387  0.135701   
1022 -0.941375  1.132169  0.161808  0.806569  0.215016 -0.355488  0.473222   
1023 -0.433659 -0.003781 -0.613696  0.748913 -0.139859 -0.269490  0.388150   
1024 -0.304051  0.620155 -0.447389  0.757313 -0.014383 -0.253349  0.284061   
1025  0.062219  0.099291 -0.811908  0.945855  0.127831 -0.207792 -0.307143   

            f8        f9       f10  ...       f47       f48    

In [16]:
# Gunakan label abstrak_id dari dataset TF-IDF berita
embedding_df['abstrak_id'] = df['abstrak_id'].values  

## Simpan Hasil ke CSV (Opsional)
Simpan `embedding_df` ke file CSV untuk digunakan di proses selanjutnya.


In [17]:
# Simpan DataFrame fitur dokumen ke CSV (opsional)
embedding_df.to_csv('pta_doc_embeddings.csv', index=False, encoding='utf-8')
print('Disimpan ke pta_doc_embeddings.csv')


Disimpan ke pta_doc_embeddings.csv


In [18]:
embedding_df

,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,...,f48,f49,f50,f51,f52,f53,f54,f55,f56,abstrak_id
0,-0.187877,-0.125373,-0.482323,0.548948,-0.051382,-0.167641,0.016427,-0.627746,-0.535999,-0.312082,...,0.118458,-0.381231,0.332271,0.231465,0.126711,-0.174610,0.215755,0.167780,0.291902,Aplikasi nyata pemanfaatan teknologi informasi...
1,-0.557555,0.154534,-0.605653,0.569248,-0.223671,0.012987,0.120140,-0.810628,-0.384784,-0.507085,...,0.170115,-0.286896,0.180644,0.645479,0.099420,-0.270224,0.397974,-0.098469,0.399390,Tujuan penelitian ini adalah untuk mengetahui ...
2,-0.029169,0.080073,-0.849571,0.747365,0.095475,-0.209035,-0.108454,-0.615794,-0.749767,-0.268187,...,0.346495,-0.533432,0.407178,0.144215,0.347277,-0.348035,0.256151,-0.094244,-0.023836,"ABSTRAK\n\nSatiyah, Pengaruh Faktor-faktor Pel..."
3,-0.038177,0.686166,-0.818487,0.994374,0.099712,-0.285629,0.119406,-0.615550,-0.671911,-0.337766,...,0.335666,-0.878491,0.812750,0.134596,0.320161,-0.402400,0.290609,-0.108851,-0.013740,Abstrak\n\nPenelitian ini menggunakan metode k...
4,-0.580001,0.438209,0.139573,0.579606,0.347730,-0.519288,0.203503,-0.406267,-0.372078,-0.394890,...,0.360857,-0.482358,-0.185161,0.322435,0.183158,0.123966,0.164025,0.416176,0.129946,Hasil dari penelitian ini dari perhitungan Cre...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1021,0.075879,0.546321,-0.568065,1.203466,0.393896,-0.622387,0.135701,-0.418982,-0.768816,-0.200402,...,0.510670,-1.148646,0.592647,-0.138525,0.644567,-0.291481,0.004952,0.002223,-0.288190,"ABSTRAK\nUswatun Hasanah, 160211100291, Pengar..."
1022,-0.941375,1.132169,0.161808,0.806569,0.215016,-0.355488,0.473222,-0.787474,-0.109415,-0.896964,...,0.415046,-0.757407,0.140370,0.697530,0.142594,0.185047,0.451360,0.279743,0.423413,Tujuan penelitian ini adalah untuk mengetahui ...
1023,-0.433659,-0.003781,-0.613696,0.748913,-0.139859,-0.269490,0.388150,-0.813797,-0.818169,-0.454390,...,-0.060801,-0.625039,0.613808,0.405506,0.083303,-0.394385,0.306843,0.402101,0.476292,ABSTRAK\nPenelitian ini bertujuan: (1) Untuk m...
1024,-0.304051,0.620155,-0.447389,0.757313,-0.014383,-0.253349,0.284061,-0.576284,-0.375350,-0.435031,...,0.289891,-0.757118,0.478434,0.365697,0.113789,-0.122563,0.289026,-0.028126,0.163461,ABSTRAK\nTujuan dari penelitian ini adalah unt...


In [19]:
embedding_df.shape

(1026, 57)